# Projeto MEGA - Análise Semanal Automatizada

Este notebook identifica automaticamente a categoria e filial de maior impacto e consolida os dados em uma **série temporal semanal**, utilizando bibliotecas oficiais para detecção de feriados.

In [5]:
# Instalação de biblioteca de feriados (se necessário)
try:
    import holidays
except ImportError:
    !pip install holidays
    import holidays

import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Carga de dados
target_file = 'data/database_final.csv'
df = pd.read_csv(target_file, low_memory=False)
df['DATA_ATEND'] = pd.to_datetime(df['DATA_ATEND'])

print(f"Tamanho do DataFrame original: {df.shape}")

Tamanho do DataFrame original: (556198, 13)


## 1. Seleção Automática de Categoria e Filial
O código identifica a combinação com o **maior número de registros** para maximizar a qualidade do treino.

In [6]:
# Identificando o par Categoria/Filial com maior volume
top_alvo = df.groupby(['CATEGORIA', 'FILIAL']).size().reset_index(name='CONTAGEM')
top_alvo = top_alvo.sort_values(by='CONTAGEM', ascending=False).iloc[0]

cat_alvo = top_alvo['CATEGORIA']
filial_alvo = top_alvo['FILIAL']

print(f"Alvo Selecionado Automaticamente:\n- Categoria: {cat_alvo}\n- Filial: {filial_alvo}\n- Registros: {top_alvo['CONTAGEM']}")

# Filtragem Dinâmica
df_filtrado = df[(df['CATEGORIA'] == cat_alvo) & (df['FILIAL'] == filial_alvo)].copy()

Alvo Selecionado Automaticamente:
- Categoria: Temperos & Condimentos
- Filial: SHOPPING
- Registros: 91755


## 2. Agregação Semanal e Detecção de Feriados Dinâmica

In [7]:
# Agregação Semanal
df_semanal = df_filtrado.set_index('DATA_ATEND').resample('W')['FATUR_VENDA'].sum().reset_index()
df_semanal.columns = ['DATA_INICIO_SEMANA', 'DEMANDA_ALVO']

# Configuração do calendário de feriados do Brasil
br_holidays = holidays.Brazil()

def verificar_feriado_robusto(row):
    data_fim = row['DATA_INICIO_SEMANA']
    data_ini = data_fim - pd.Timedelta(days=6)
    # Gera todos os dias da semana e verifica se algum é feriado no Brasil
    periodo = pd.date_range(start=data_ini, end=data_fim)
    for data in periodo:
        if data in br_holidays:
            return 1
    return 0

df_semanal['MES'] = df_semanal['DATA_INICIO_SEMANA'].dt.month
df_semanal['TEM_FERIADO'] = df_semanal.apply(verificar_feriado_robusto, axis=1)

# Dataset Final para Treinamento (Removendo a data)
dataset_treino = df_semanal.drop(columns=['DATA_INICIO_SEMANA'])

print(f"Série semanal gerada com {df_semanal.shape[0]} semanas.")
dataset_treino.head()

Série semanal gerada com 53 semanas.


,DEMANDA_ALVO,MES,TEM_FERIADO
0,7047.87,1,1
1,8431.63,1,0
2,8233.01,1,0
3,7691.11,1,0
4,9739.67,2,0


## 3. Definição de Níveis Semanais (Thresholds)

In [8]:
q33 = df_semanal['DEMANDA_ALVO'].quantile(0.33)
q66 = df_semanal['DEMANDA_ALVO'].quantile(0.66)

print("=== LIMITES SEMANAIS CALCULADOS ===")
print(f"Demanda Baixa: Até R$ {q33:.2f}")
print(f"Demanda Média: Entre R$ {q33:.2f} e R$ {q66:.2f}")
print(f"Demanda Alta: Acima de R$ {q66:.2f}")

=== LIMITES SEMANAIS CALCULADOS ===
Demanda Baixa: Até R$ 8739.35
Demanda Média: Entre R$ 8739.35 e R$ 10066.61
Demanda Alta: Acima de R$ 10066.61
